In [ ]:
import os
import getpass
import random

from dotenv import load_dotenv

load_dotenv(".env")

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage

from typing import TypedDict, List, Annotated, Any
from pydantic import BaseModel, Field


In [ ]:
# LLMs are initialized in the Gemini config cell after GOOGLE_API_KEY is ready.
llm_generate = None
llm_grader = None


## RAG

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Gemini API config
# If GOOGLE_API_KEY is not set, router, memory, and retrieval tests still run.

GEMINI_MODEL = "gemini-2.5-flash"

if os.environ.get("GOOGLE_API_KEY"):
    llm_generate = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        temperature=0.2,
        max_retries=2,
    )

    llm_grader = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        temperature=0,
        max_retries=2,
    )
else:
    print("GOOGLE_API_KEY is not set. RAG generation is skipped, but router/memory/retrieval tests can run.")
    llm_generate = None
    llm_grader = None

# Optional local model fallback.
# from langchain_ollama import ChatOllama
# llm_generate = ChatOllama(model="qwen2.5:7b", temperature=0.3)
# llm_grader = ChatOllama(model="qwen2.5:7b", temperature=0)


In [ ]:
# =============================================================================
# RAG CONFIG
# =============================================================================

import os
import random
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

HF_CACHE_DIR = PROJECT_ROOT / ".cache" / "huggingface"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(HF_CACHE_DIR / "sentence_transformers")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Retrieval profile:
# - legacy: your original compact Chroma DB, currently stronger for broad QA.
# - hybrid: the newer Kaggle hybrid QA index, useful for further experiments.
RETRIEVER_PROFILE = os.getenv("QA_RETRIEVER_PROFILE", "legacy").strip().lower()

if RETRIEVER_PROFILE == "hybrid":
    default_index_dir = PROJECT_ROOT / "chroma_db_qa_hybrid"
    default_collection_name = "qa_hybrid_chunks"
else:
    default_index_dir = PROJECT_ROOT / "chroma_db"
    default_collection_name = "langchain"

QA_INDEX_DIR = Path(os.getenv("QA_CHROMA_DIR", default_index_dir))
COLLECTION_NAME = os.getenv("QA_CHROMA_COLLECTION", default_collection_name)
QA_TEST_INDEX_DIR = PROJECT_ROOT / "chroma_db_qa_test"

if not QA_INDEX_DIR.exists() and QA_TEST_INDEX_DIR.exists():
    print(f"[RAG] QA index not found at {QA_INDEX_DIR}")
    print(f"[RAG] Falling back to test index: {QA_TEST_INDEX_DIR}")
    QA_INDEX_DIR = QA_TEST_INDEX_DIR
    if "QA_CHROMA_COLLECTION" not in os.environ:
        COLLECTION_NAME = "qa_chunks"

PERSIST_DIR = str(QA_INDEX_DIR)
EMBED_MODEL = "intfloat/multilingual-e5-base"
DEVICE = os.getenv("QA_RETRIEVER_DEVICE", "cpu")
TOP_K = 5
FETCH_K = int(os.getenv("QA_FETCH_K", "50"))
LEXICAL_WEIGHT = float(os.getenv("QA_LEXICAL_WEIGHT", "0.06"))

# Final score is vector distance adjusted by lexical topic matching. Lower is
# better. Keep empty by default until you calibrate a threshold.
raw_retrieval_max_score = os.getenv("QA_RETRIEVAL_MAX_SCORE", "").strip()
RETRIEVAL_MAX_SCORE = float(raw_retrieval_max_score) if raw_retrieval_max_score else None

MAX_RETRIES = 3
SEED = 42

random.seed(SEED)


In [ ]:
# =============================================================================
# QA RETRIEVER
# =============================================================================

from src.retrieval.qa_retriever import QaRetriever

qa_retriever = QaRetriever(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
    model_name=EMBED_MODEL,
    device=DEVICE,
    lexical_weight=LEXICAL_WEIGHT,
)

print(f"[RAG] Profile: {RETRIEVER_PROFILE}")
print(f"[RAG] Loaded QA retriever from: {PERSIST_DIR}")
print(f"[RAG] Collection: {COLLECTION_NAME}")
print(f"[RAG] Device: {DEVICE}")
print(f"[RAG] Fetch K: {FETCH_K}")
print(f"[RAG] Lexical weight: {LEXICAL_WEIGHT}")


In [ ]:
# =============================================================================
# RETRIEVAL PREVIEW HELPERS
# =============================================================================

last_retrieved_chunks = []


def retrieve(query, k=TOP_K, fetch_k=FETCH_K, max_score=RETRIEVAL_MAX_SCORE):
    """
    Retrieve LangChain Documents from the hybrid QA Chroma index.

    The retriever internally keeps score/rank metadata for debugging, while the
    graph still receives plain LangChain Document objects like before.
    """

    global last_retrieved_chunks

    retrieved_chunks = qa_retriever.retrieve(
        query=query,
        top_k=k,
        fetch_k=fetch_k,
        max_score=max_score,
    )
    last_retrieved_chunks = retrieved_chunks

    return [chunk.document for chunk in retrieved_chunks]


def show_results(query, k=TOP_K, fetch_k=FETCH_K, max_score=RETRIEVAL_MAX_SCORE):
    """Print retrieval results so you can inspect topic/keyword/score quality."""

    retrieved_chunks = qa_retriever.retrieve(
        query=query,
        top_k=k,
        fetch_k=fetch_k,
        max_score=max_score,
    )
    qa_retriever.print_results(
        query=query,
        retrieved_chunks=retrieved_chunks,
    )

    return [chunk.document for chunk in retrieved_chunks]


In [ ]:
# =============================================================================
# CONTEXT FORMATTER
# =============================================================================


def build_context(docs):
    """Format retrieved documents into a readable context block."""

    context_blocks = []

    for index, doc in enumerate(docs, start=1):
        metadata = getattr(doc, "metadata", {}) or {}
        context_blocks.append(
            "\n".join([
                f"[Document {index}]",
                f"Chunk Type: {metadata.get('chunk_type', '')}",
                f"Retrieval Anchor: {metadata.get('retrieval_anchor', '')}",
                f"Canonical Topic: {metadata.get('canonical_topic', '')}",
                f"Keyword: {metadata.get('keyword', '')}",
                f"Normalized Keyword: {metadata.get('normalized_keyword', '')}",
                f"Topic: {metadata.get('topic', '')}",
                f"Category: {metadata.get('category', '')}",
                f"Question Type: {metadata.get('question_type', '')}",
                f"Question: {metadata.get('question', '')}",
                f"Trace: image_id={metadata.get('image_id', '')}",
                "",
                doc.page_content,
            ])
        )

    return "\n\n".join(context_blocks)


In [ ]:
# =============================================================================
# RAG PIPELINE
# =============================================================================

import time


def rag_answer(query, k=TOP_K):
    """Retrieve documents for the graph's RAG node."""

    docs = retrieve(query, k=k)

    if not docs:
        print(
            "[RAG] No strong documents found. "
            "Check chroma_db_qa_hybrid or lower QA_RETRIEVAL_MAX_SCORE."
        )

    return docs


In [ ]:
# Old vectorstore-based rag_answer was replaced by QaRetriever in the cells above.


## LLM Generate

In [ ]:
class GraphState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    intent: str
    route_reason: str
    transformed_question: str
    documents: List[Any]
    answer: str
    user_id: str
    # Long-term memory loaded from user_memories.json.
    user_preferences: str
    retry_count: int


In [ ]:
# Prompt

template = """You are a professional assistant for Vietnamese culture and tourism.
Rules:
- Respond only in Vietnamese.
- Use only the provided documents to answer the user question.
- If the question asks for a definition, start with a short definition grounded in the documents.
- Do not add outside knowledge. If the documents are insufficient, say: "Toi khong co du thong tin de tra loi cau hoi nay".
- PERSONALIZATION RULE: You are provided with the user's historical cultural/historical preferences (User Preferences). Use this information ONLY to adjust your tone, focus on the aspects they care about within the provided documents, or prioritize organizing the information in a way they prefer. Do NOT use User Preferences as a fact source to invent information outside the Documents.

User Preferences (For personalization style and focus):
{user_preferences}

Documents:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Post-processing
def format_docs(docs):
    return build_context(docs)

# Chain
# Keep this cell runnable even when the Gemini API key is temporarily skipped.
rag_chain = None
if llm_generate is not None:
    rag_chain = prompt | llm_generate | StrOutputParser()


## Generate Node

In [ ]:
def generate(state: GraphState):
    """Generate an answer from retrieved documents."""

    print("---GENERATE---")
    documents = state["documents"]
    messages = state["messages"]
    user_preferences = state.get("user_preferences", "Chưa có thông tin sở thích.")
    retry_count = state.get("retry_count", 0) + 1
    original_question = messages[-1].content

    if rag_chain is None:
        generation = (
            "Chưa cấu hình LLM để sinh câu trả lời RAG. "
            "Hãy thiết lập GOOGLE_API_KEY hoặc chỉ test các intent memory/recommendation trước."
        )
        print(generation)
        return {"answer": generation, "retry_count": retry_count}

    generation = rag_chain.invoke({
        "context": format_docs(documents),
        "question": original_question,
        "user_preferences": user_preferences,
    })

    print(f"Generate attempt: {retry_count}/{MAX_RETRIES}")
    print(f"Original question: {original_question}")
    print(f"Generated answer: {generation}")

    return {"answer": generation, "retry_count": retry_count}


### Hallucination Grader

In [ ]:
class GradeHallucinations(BaseModel):
    """Binary groundedness score for an answer against retrieved documents."""
    binary_score: bool = Field(
        description="True if all main claims in the answer are supported by the documents or equivalent paraphrases; False if the answer adds unsupported or contradictory factual claims."
    )
    reasoning: str = Field(
        description="Short Vietnamese explanation for the score."
    )


def check_hallucination(state):
    """
    Check groundedness only. This function does not judge answer usefulness or relevance.
    """
    print("=== Check groundedness / hallucination ===")
    structured_llm_grader = llm_grader.with_structured_output(GradeHallucinations)

    system_prompt = """You are a groundedness grader for a RAG system.
Your only task is to check whether the ANSWER is supported by the DOCUMENTS.

Rules:
- Return True if every main claim in the answer appears in the documents or is an equivalent paraphrase.
- Return True if the answer uses synonyms/paraphrases without adding new factual information.
- Return False only if the answer adds factual information not present in the documents or contradicts the documents.
- Do not grade usefulness, completeness, or whether the answer directly answers the user question.
- If the answer is incomplete but all claims it makes are supported, return True.
- Write the reasoning in Vietnamese."""

    hallucination_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "DOCUMENTS:\n\n{documents}\n\nANSWER:\n{generation}"),
        ]
    )

    hallucination_grader = hallucination_prompt | structured_llm_grader

    documents = state["documents"]
    generation = state.get("answer", "")
    docs_text = format_docs(documents)

    try:
        score = hallucination_grader.invoke({
            "documents": docs_text,
            "generation": generation
        })
    except Exception as e:
        print(f"Hallucination grader API error: {e}")
        return False

    print(f"Groundedness reasoning: {score.reasoning}")
    print(f"Groundedness score (binary_score): {score.binary_score}")
    if score.binary_score:
        print("=> Pass. The answer is supported by the documents.")
    else:
        print("=> Fail. The answer contains unsupported information.")
    return score.binary_score


In [ ]:
# LLM-as-a-Judge schema for answer usefulness
class GradeAnswer(BaseModel):
    """Evaluate whether the answer solves the original question."""
    is_relevant: bool = Field(
        description="True if the answer directly and correctly addresses the original question; otherwise False."
    )
    reasoning: str = Field(
        description="Short Vietnamese explanation, 1-2 sentences."
    )


def evaluate_answer(state: GraphState):
    print("---EVALUATE ANSWER USEFULNESS---")
    messages = state.get("messages")
    original_question = messages[-1].content
    transformed_question = state.get("transformed_question", "")
    answer = state.get("answer", "")
    print("Answer:", answer)

    structured_llm_judge = llm_grader.with_structured_output(GradeAnswer)

    system_prompt = """You are a strict answer-quality grader.
Your task is to evaluate whether the final answer directly and correctly solves the user's ORIGINAL QUESTION.

Rules:
1. Grade only against the original question.
2. The rewritten question, if any, is only retrieval context and must not be the grading target.
3. If the answer says there is not enough information and the documents are genuinely insufficient, consider it acceptable.
4. Write the reasoning in Vietnamese."""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", """
        ORIGINAL QUESTION: {original_question}
        REWRITTEN QUESTION FOR RETRIEVAL: {transformed_question}

        FINAL ANSWER:
        {answer}
        """)
    ])

    evaluator_chain = prompt | structured_llm_judge

    result = evaluator_chain.invoke({
        "original_question": original_question,
        "transformed_question": transformed_question,
        "answer": answer
    })

    print(f"Usefulness reasoning: {result.reasoning}")
    print(f"Usefulness score (is_relevant): {result.is_relevant}")

    return result.is_relevant


In [ ]:
def rag_node(state: GraphState):
    """Retrieve documents using the transformed query, with a safe fallback."""

    print("\n=== RUN RAG ===")
    search_query = state.get("transformed_question")

    if not search_query:
        messages = state.get("messages", [])
        search_query = messages[-1].content if messages else ""

    retrieved_docs = rag_answer(search_query)

    print(f"Search query: '{search_query}'")
    print(f"Retrieved documents: {len(retrieved_docs)}")
    print("================\n")

    return {"documents": retrieved_docs}


In [ ]:
def check_hallucination_and_evaluate(state: GraphState):
    if state.get("retry_count", 0) >= MAX_RETRIES:
        print("---MAX RETRIES REACHED, END GRAPH---")
        return "max_retries"

    if llm_grader is None:
        print("---SKIP GRADER: llm_grader is not configured---")
        return "max_retries"

    is_grounded = check_hallucination(state)
    if not is_grounded:
        return "is_hallucinated"

    is_useful = evaluate_answer(state)
    if is_useful:
        return "is_useful"
    return "is_not_useful"


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def transform_query(state: GraphState):
    """Rewrite the latest question for retrieval when an LLM is available."""

    print("\n--- [NODE] DATASET-ALIGNED TRANSFORM QUERY ---")

    messages = state.get("messages", [])
    user_preferences = state.get("user_preferences", "Chưa có thông tin sở thích.")
    retry_count = state.get("retry_count", 0)
    current_search_query = state.get("transformed_question", "")
    original_question = messages[-1].content if messages else ""

    # When Gemini is not configured, keep the graph testable by using the raw question.
    if llm_generate is None:
        print("LLM rewrite is not configured. Using the original question for retrieval.")
        return {"transformed_question": original_question}

    system_prompt = """You are a RAG query optimization expert for Vietnamese culture and tourism.
Your mission is to rewrite the user's latest question into one search query for a Vector Database/BM25 engine.

Dataset categories:
['Kiến trúc', 'Ẩm thực', 'Phong cảnh', 'Trang phục', 'Đời sống hằng ngày', 'Văn hóa dân gian', 'Lễ hội', 'Trò chơi dân gian', 'Thể thao truyền thống', 'Thủ công mỹ nghệ', 'Nhạc cụ', 'Giao thông']

Rules:
- Resolve references from chat history, such as 'nó', 'món này', 'thứ này', or 'trang phục đó'.
- Remove conversational filler and fix obvious typos.
- Add useful dataset keywords only when they clarify the query.
- If this is a retry, avoid repeating the exact previous query.
- Return only the rewritten Vietnamese search query. No explanation."""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        *messages[:-1],
        ("human", """User Historical Preferences: {user_preferences}
Previous Query Used: {current_search_query}
Current Retry Count: {retry_count}
User's Latest Question: {original_question}
REWRITTEN QUERY FOR DATABASE:"""),
    ])

    chain = prompt | llm_generate | StrOutputParser()
    better_query = chain.invoke({
        "user_preferences": user_preferences,
        "current_search_query": current_search_query,
        "retry_count": retry_count,
        "original_question": original_question,
    }).strip()

    print(f"Attempt: {retry_count + 1}")
    print(f"Original question: '{original_question}'")
    print(f"Dataset search query: '{better_query}'")
    print("------------------------------------------\n")

    return {"transformed_question": better_query}


In [ ]:
import json

from langchain_core.messages import AIMessage

import importlib
import src.routing.routing as routing_module

importlib.reload(routing_module)

from src.routing.routing import (
    build_grounded_recommendation_message,
    build_langgraph_config,
    build_memory_summary,
    build_preference_saved_message,
    build_recommendation_message,
    build_recommendation_query,
    classify_intent,
    dump_memory_json,
    extract_memory_updates_from_text,
    load_memory_json,
    load_user_memory,
    merge_memory,
    save_user_memory,
    update_memory_from_retrieved_documents,
)

MEMORY_FILE = "user_memories.json"


def route_intent_node(state: GraphState):
    """Classify the latest user message before deciding whether RAG is needed."""

    print("\n--- [NODE] ROUTE INTENT ---")
    messages = state.get("messages", [])
    latest_message = messages[-1].content if messages else ""

    decision = classify_intent(latest_message)
    print(f"Intent: {decision.intent}")
    print(f"Reason: {decision.reason}")

    return {
        "intent": decision.intent,
        "route_reason": decision.reason,
    }


def route_after_intent(state: GraphState):
    """Map intent labels to graph branches."""

    intent = state.get("intent", "rag_question")

    if intent in {"rag_question", "followup_question"}:
        return "needs_rag"

    return "no_rag_needed"


def non_rag_intent_node(state: GraphState):
    """Handle memory/recommendation/chitchat intents without calling the answer LLM."""

    print("\n--- [NODE] NON-RAG INTENT HANDLER ---")
    intent = state.get("intent", "chitchat")
    user_id = state.get("user_id", "anonymous")
    messages = state.get("messages", [])
    latest_message = messages[-1].content if messages else ""
    current_memory = load_memory_json(state.get("user_preferences", ""))

    if intent == "preference_update":
        memory_updates = extract_memory_updates_from_text(latest_message)
        updated_memory = merge_memory(
            current_memory,
            memory_updates,
            evidence_text=latest_message,
        )
        save_user_memory(MEMORY_FILE, user_id, updated_memory)
        answer = build_preference_saved_message(updated_memory)
        return {
            "answer": answer,
            "user_preferences": dump_memory_json(updated_memory),
            "messages": [AIMessage(content=answer)],
        }

    if intent == "memory_query":
        answer = build_memory_summary(current_memory)
        documents = []
    elif intent == "recommendation_request":
        recommendation_query = build_recommendation_query(current_memory)
        if recommendation_query:
            print(f"Recommendation retrieval query: {recommendation_query}")
            retrieved_chunks = qa_retriever.retrieve(
                query=recommendation_query,
                top_k=3,
                fetch_k=FETCH_K,
            )
            answer = build_grounded_recommendation_message(
                current_memory,
                retrieved_chunks,
            )
            documents = [chunk.document for chunk in retrieved_chunks]
        else:
            answer = build_recommendation_message(current_memory)
            documents = []
    elif intent == "out_of_scope":
        answer = (
            "Mình chỉ có dữ liệu về văn hóa Việt Nam trong dataset hiện tại. "
            "Bạn thử hỏi về ẩm thực, lễ hội, trang phục, giao thông hoặc thủ công mỹ nghệ nhé."
        )
        documents = []
    else:
        answer = "Chào bạn, mình sẵn sàng hỗ trợ các câu hỏi về văn hóa Việt Nam."
        documents = []

    print(f"Answer: {answer}")
    return {
        "answer": answer,
        "documents": documents,
        "messages": [AIMessage(content=answer)],
    }


In [ ]:
# Legacy LLM memory loader was replaced by the rule-based memory node below.
# Keep this cell as a note so the notebook execution order remains stable.


In [ ]:
# Legacy LLM memory updater was replaced by the rule-based memory node below.
# The active implementation is in the next cell: RULE-BASED MEMORY OVERRIDES.


In [ ]:
# =============================================================================
# RULE-BASED MEMORY OVERRIDES
# =============================================================================

def load_memory_node(state: GraphState):
    """Load long-term memory for the current user."""

    print("
--- [NODE] LOAD LONG-TERM MEMORY ---")
    user_id = state.get("user_id", "anonymous")
    memory = load_user_memory(MEMORY_FILE, user_id)
    user_preferences = dump_memory_json(memory)

    print(f"User ID: {user_id}")
    print(f"Loaded memory: {user_preferences}")
    print("------------------------------------
")

    return {"user_preferences": user_preferences}


def update_memory_node(state: GraphState):
    """Keep RAG questions out of long-term preference memory."""

    print("
--- [NODE] KEEP MEMORY UNCHANGED AFTER RAG ---")
    current_memory = load_memory_json(state.get("user_preferences", ""))
    current_preferences_json = dump_memory_json(current_memory)

    print("RAG turn finished. Long-term preferences are updated only when the user explicitly states a preference.")
    print(current_preferences_json)
    print("------------------------------------------------
")

    return {
        "user_preferences": current_preferences_json,
        "messages": [AIMessage(content=state.get("answer", ""))],
    }


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(GraphState)

workflow.add_node("load_memory", load_memory_node)
workflow.add_node("route_intent", route_intent_node)
workflow.add_node("non_rag_intent", non_rag_intent_node)
workflow.add_node("transform", transform_query)
workflow.add_node("rag", rag_node)
workflow.add_node("generate", generate)
workflow.add_node("update_memory", update_memory_node)

workflow.add_edge(START, "load_memory")
workflow.add_edge("load_memory", "route_intent")
workflow.add_conditional_edges(
    "route_intent",
    route_after_intent,
    {
        "needs_rag": "transform",
        "no_rag_needed": "non_rag_intent",
    },
)
workflow.add_edge("non_rag_intent", END)
workflow.add_edge("transform", "rag")
workflow.add_edge("rag", "generate")
workflow.add_edge("update_memory", END)

workflow.add_conditional_edges(
    "generate",
    check_hallucination_and_evaluate,
    {
        "is_hallucinated": "generate",
        "is_useful": "update_memory",
        "is_not_useful": "transform",
        "max_retries": END,
    },
)

memory_checkpointer = MemorySaver()
app = workflow.compile(checkpointer=memory_checkpointer)


## Ordered Smoke Tests

Run these cells from top to bottom after the graph build cell.


In [ ]:
from langchain_core.messages import HumanMessage

TEST_USER_ID = "test_memory_agent"
TEST_THREAD_ID = build_langgraph_config(TEST_USER_ID, "test_router")["configurable"]["thread_id"]
TEST_CONFIG = build_langgraph_config(TEST_USER_ID, "test_router")


def print_test_state(label, state):
    """Print only the fields needed to verify routing and memory."""

    print("=" * 80)
    print(label)
    print("intent:", state.get("intent"))
    print("route_reason:", state.get("route_reason"))
    print("answer:", state.get("answer"))
    print("transformed_question:", state.get("transformed_question"))
    print("documents:", len(state.get("documents", [])))
    print("user_preferences:", state.get("user_preferences"))


In [ ]:
state_preference = app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích lễ hội và ẩm thực")],
        "user_id": TEST_USER_ID,
    },
    config=TEST_CONFIG,
)

print_test_state("TEST 1 - preference_update", state_preference)


In [ ]:
state_memory_query = app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích gì?")],
        "user_id": TEST_USER_ID,
    },
    config=TEST_CONFIG,
)

print_test_state("TEST 2 - memory_query", state_memory_query)


In [ ]:
state_recommendation = app.invoke(
    {
        "messages": [HumanMessage(content="Gợi ý cho tôi chủ đề hay")],
        "user_id": TEST_USER_ID,
    },
    config=TEST_CONFIG,
)

print_test_state("TEST 3 - recommendation_request", state_recommendation)


In [ ]:
# Direct retrieval test. This does not call Gemini.
show_results("So sánh bánh chưng và bánh tét", k=5)


In [ ]:
# RAG path smoke test.
# Without GOOGLE_API_KEY, this should retrieve documents and stop with a clear LLM-config message.
# With GOOGLE_API_KEY configured, this will continue to generation/evaluation.
state_rag = app.invoke(
    {
        "messages": [HumanMessage(content="Xe máy là gì?")],
        "user_id": TEST_USER_ID,
    },
    config=build_langgraph_config(TEST_USER_ID, "test_rag"),
)

print_test_state("TEST 5 - rag_question", state_rag)


In [ ]:
# Optional cleanup: remove the temporary test user from user_memories.json.
memory_path = Path(MEMORY_FILE)
if memory_path.exists():
    memory_db = json.loads(memory_path.read_text(encoding="utf-8"))
    memory_db.pop(TEST_USER_ID, None)
    memory_path.write_text(json.dumps(memory_db, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Removed temporary test user: {TEST_USER_ID}")


## Baseline Showcase

Run these cells after the smoke tests when you want to demo the baseline end to end.


In [ ]:
# Showcase setup: reset demo users so the output is easy to read.
from pathlib import Path
import json

SHOWCASE_USER_ID = "showcase_main"
SHOWCASE_USER_A = "showcase_user_a"
SHOWCASE_USER_B = "showcase_user_b"
SHOWCASE_USER_IDS = [SHOWCASE_USER_ID, SHOWCASE_USER_A, SHOWCASE_USER_B]


def remove_demo_users(user_ids):
    """Remove demo users from JSON memory before/after showcase runs."""

    memory_path = Path(MEMORY_FILE)
    if not memory_path.exists():
        return

    memory_db = json.loads(memory_path.read_text(encoding="utf-8"))
    for user_id in user_ids:
        memory_db.pop(user_id, None)

    memory_path.write_text(
        json.dumps(memory_db, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


remove_demo_users(SHOWCASE_USER_IDS)
SHOWCASE_CONFIG = build_langgraph_config(SHOWCASE_USER_ID, "main_demo")

print("Showcase users reset.")
print("Main user thread:", SHOWCASE_CONFIG["configurable"]["thread_id"])


In [ ]:
# Showcase 1: explicit preference is saved, then memory can answer it back.
showcase_preference = app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích lễ hội và ẩm thực")],
        "user_id": SHOWCASE_USER_ID,
    },
    config=SHOWCASE_CONFIG,
)
print_test_state("SHOWCASE 1A - save preference", showcase_preference)

showcase_memory = app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích gì?")],
        "user_id": SHOWCASE_USER_ID,
    },
    config=SHOWCASE_CONFIG,
)
print_test_state("SHOWCASE 1B - read memory", showcase_memory)


In [ ]:
# Showcase 2: recommendation uses saved preference and retrieved dataset chunks.
showcase_recommendation = app.invoke(
    {
        "messages": [HumanMessage(content="Gợi ý cho tôi vài chủ đề phù hợp")],
        "user_id": SHOWCASE_USER_ID,
    },
    config=SHOWCASE_CONFIG,
)
print_test_state("SHOWCASE 2 - grounded recommendation", showcase_recommendation)


In [ ]:
# Showcase 3: RAG question goes through transform -> retrieve -> Gemini generation -> grading.
showcase_rag = app.invoke(
    {
        "messages": [HumanMessage(content="Xe máy là gì?")],
        "user_id": SHOWCASE_USER_ID,
    },
    config=build_langgraph_config(SHOWCASE_USER_ID, "rag_demo"),
)
print_test_state("SHOWCASE 3 - RAG generation", showcase_rag)


In [ ]:
# Showcase 4: two users keep separate long-term memory and separate thread history.
user_a_config = build_langgraph_config(SHOWCASE_USER_A, "shared_demo")
user_b_config = build_langgraph_config(SHOWCASE_USER_B, "shared_demo")

app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích lễ hội")],
        "user_id": SHOWCASE_USER_A,
    },
    config=user_a_config,
)
app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích giao thông")],
        "user_id": SHOWCASE_USER_B,
    },
    config=user_b_config,
)

user_a_memory = app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích gì?")],
        "user_id": SHOWCASE_USER_A,
    },
    config=user_a_config,
)
user_b_memory = app.invoke(
    {
        "messages": [HumanMessage(content="Tôi thích gì?")],
        "user_id": SHOWCASE_USER_B,
    },
    config=user_b_config,
)

print("USER A thread:", user_a_config["configurable"]["thread_id"])
print_test_state("SHOWCASE 4A - user A memory", user_a_memory)
print("USER B thread:", user_b_config["configurable"]["thread_id"])
print_test_state("SHOWCASE 4B - user B memory", user_b_memory)


In [ ]:
# Showcase cleanup: remove demo users after recording the result.
remove_demo_users(SHOWCASE_USER_IDS)
print("Showcase users removed:", ", ".join(SHOWCASE_USER_IDS))
